In [5]:
import os
import re
import ast
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# Load API keys from root .env
load_dotenv(dotenv_path="../../../.env")
if not os.getenv("OPENROUTER_API_KEY"):
    load_dotenv()

# Store all available keys in a list
api_keys = []
key1 = os.getenv("OPENROUTER_API_KEY")
key2 = os.getenv("OPENROUTER_API_KEY_NEW")

if key1: api_keys.append(key1)
if key2: api_keys.append(key2)

if not api_keys:
    raise ValueError("No API keys found! Please verify your .env file.")

# Set evaluation model (Starting with GPT-4o-mini)
EVAL_MODEL = "x-ai/grok-4.3"

print(f"Environment ready. Loaded {len(api_keys)} API keys.")
print(f"Evaluation Model set to: {EVAL_MODEL}")

Environment ready. Loaded 2 API keys.
Evaluation Model set to: x-ai/grok-4.3


In [6]:
INPUT_FILE = "gpt5.6_luna_specialist_full_dataset.csv"
OUTPUT_FILE = "gpt5.6_luna_full_specialist_evaluated_by_grok_4_3.csv"
BACKUP_FILE = "backup_" + OUTPUT_FILE

print(f"Loading generated specialist dataset from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

def extract_target_specialist(row):
    """Extracts target specialist entity from available columns."""
    if "target_specialist" in row and pd.notna(row["target_specialist"]):
        return str(row["target_specialist"]).strip()
    val = row.get("Specialist")
    if pd.isna(val): 
        return None
    val_str = str(val).strip()
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            parsed = ast.literal_eval(val_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                return str(parsed[0]).strip()
        except Exception:
            pass
    return val_str

df["eval_target_specialist"] = df.apply(extract_target_specialist, axis=1)
df = df.dropna(subset=["text", "modified_sentence", "eval_target_specialist"]).reset_index(drop=True)
print(f"Loaded {len(df)} total rows ready for fresh evaluation.")

Loading generated specialist dataset from gpt5.6_luna_specialist_full_dataset.csv...
Loaded 650 total rows ready for fresh evaluation.


In [7]:
active_key_index = 0

PROMPT_TEMPLATE = """
You are a medical verification assistant.

Determine whether the extracted medical entity is hallucinated with respect to the given text.

Text:
{text}

Extracted Entity:
{target_entity}

If the extracted entity is hallucinated, output:
ANSWER: 1

Otherwise, output:
ANSWER: 0
"""

def evaluate_entity(text, target_entity, model_name=EVAL_MODEL):
    global active_key_index
    
    prompt = PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_entity=str(target_entity).strip()
    )

    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }

    # Loop to retry with next key if rate limit or credit depletion occurs
    for _ in range(len(api_keys)):
        current_key = api_keys[active_key_index]
        
        headers = {
            "Authorization": f"Bearer {current_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        
        # 402 = Payment Required / 429 = Rate Limited
        if response.status_code in [402, 429]:
            print(f"   [!] Key {active_key_index + 1} hit a limit (Status {response.status_code}). Swapping keys...")
            active_key_index = (active_key_index + 1) % len(api_keys)
            continue
            
        response.raise_for_status()
        result = response.json()
        
        if "choices" not in result:
            raise ValueError(f"OpenRouter did not return choices: {result}")

        content = result["choices"][0]["message"]["content"].strip()
        match = re.search(r"ANSWER:\s*([01])", content, re.IGNORECASE)
        prediction = int(match.group(1)) if match else None

        return prediction, content

    raise Exception("CRITICAL ERROR: ALL API keys have reached their limits!")

In [8]:
# 1. INITIALIZE EVALUATION RUN (OR RESUME IF INTERRUPTED)
if os.path.exists(BACKUP_FILE):
    print(f"Found existing backup file: {BACKUP_FILE}")
    df_backup = pd.read_csv(BACKUP_FILE)
    
    # Check for network error strings
    error_pattern = "HTTPSConnectionPool|Connection aborted|ConnectionResetError|Max retries exceeded|Client Error"
    bad_rows = df_backup[
        df_backup['correct_raw_response'].astype(str).str.contains(error_pattern, regex=True) |
        df_backup['hallucinated_raw_response'].astype(str).str.contains(error_pattern, regex=True)
    ]
    
    if not bad_rows.empty:
        first_bad_index = bad_rows.index[0]
        print(f"   [!] Detected network errors in backup starting at row {first_bad_index + 1}.")
        print(f"   [!] Truncating corrupted rows to restore valid progress...")
        df_backup = df_backup.iloc[:first_bad_index].copy()
        df_backup.to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")

    start_index = len(df_backup)
    correct_predictions = df_backup["correct_prediction"].tolist()
    correct_raw = df_backup["correct_raw_response"].tolist()
    hall_predictions = df_backup["hallucinated_prediction"].tolist()
    hall_raw = df_backup["hallucinated_raw_response"].tolist()
    
    correct_matches = sum(1 for p in correct_predictions if p == 0.0)
    hall_matches = sum(1 for p in hall_predictions if p == 1.0)
    
    print(f"Loaded {start_index} valid rows. Resuming from row {start_index + 1}...\n")
else:
    print("Starting fresh evaluation from row 1...\n")
    start_index = 0
    correct_predictions, hall_predictions = [], []
    correct_raw, hall_raw = [], []
    correct_matches, hall_matches = 0, 0

# 2. RUN EVALUATION LOOP
print(f"Running evaluation loop using {EVAL_MODEL}...\n")

for i in range(start_index, len(df)):
    row = df.iloc[i]
    orig_text = row["text"]
    mod_text = row["modified_sentence"]
    target_specialist = row["eval_target_specialist"]

    # Evaluate Original Sentence (Expected ANSWER: 0)
    try:
        pred_orig, raw_orig = evaluate_entity(orig_text, target_specialist)
    except Exception as e:
        print(f"Row {i+1} [Original] Error: {e}")
        pred_orig, raw_orig = None, str(e)

    correct_predictions.append(pred_orig)
    correct_raw.append(raw_orig)
    if pred_orig == 0:
        correct_matches += 1

    # Evaluate Hallucinated Sentence (Expected ANSWER: 1)
    try:
        pred_hall, raw_hall = evaluate_entity(mod_text, target_specialist)
    except Exception as e:
        print(f"Row {i+1} [Hallucinated] Error: {e}")
        pred_hall, raw_hall = None, str(e)

    hall_predictions.append(pred_hall)
    hall_raw.append(raw_hall)
    if pred_hall == 1:
        hall_matches += 1

    print(f"[{i+1}/{len(df)}] Orig Pred: {pred_orig} (Exp: 0) | Hall Pred: {pred_hall} (Exp: 1)")
    
    # Auto-save backup every 50 rows
    if (i + 1) % 50 == 0:
        df_temp = df.loc[:i].copy()
        df_temp["correct_prediction"] = correct_predictions
        df_temp["correct_raw_response"] = correct_raw
        df_temp["hallucinated_prediction"] = hall_predictions
        df_temp["hallucinated_raw_response"] = hall_raw
        df_temp.to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")
        print(f"   --- Auto-saved backup at row {i+1} ---")

    time.sleep(0.5)

# 3. FINAL SAVE & ACCURACY METRICS
df["correct_prediction"] = correct_predictions
df["correct_raw_response"] = correct_raw
df["hallucinated_prediction"] = hall_predictions
df["hallucinated_raw_response"] = hall_raw

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

total_rows = len(df)
correct_accuracy = (correct_matches / total_rows) * 100 if total_rows > 0 else 0
hall_accuracy = (hall_matches / total_rows) * 100 if total_rows > 0 else 0

print("\n==============================")
print(f"Evaluator Model                : {EVAL_MODEL}")
print(f"Correct Sentence Accuracy      : {correct_accuracy:.2f}%")
print(f"Hallucinated Sentence Accuracy : {hall_accuracy:.2f}%")
print(f"Saved detailed results to      : {OUTPUT_FILE}")
print("==============================")

Starting fresh evaluation from row 1...

Running evaluation loop using x-ai/grok-4.3...

   [!] Key 1 hit a limit (Status 402). Swapping keys...
[1/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[2/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[3/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[4/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[5/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[6/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[7/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[8/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[9/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[10/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[11/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[12/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[13/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[14/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[15/650] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[16/650] Orig Pred: 0 (Exp: 0) | Hall P